## Iowa Optimization — Euclidean M (Euclidean-Weighted Distances)

This notebook runs multi-district MIP optimization experiments on the Iowa county graph using **Euclidean-weighted distances** for the tree, dist, and dag contiguity constraints.

**Euclidean M** means the Big-M parameter in the distance-based contiguity formulations is derived from weighted shortest paths, where every edge weight is the Euclidean distance between the centroids of adjacent counties (in projected km). This means the formulations prefer paths that are geographically short, not just topologically short. This mirrors the Euclidean-weighted version of the feasibility analysis.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT_DIR))

import os
import matplotlib.pyplot as plt

from src.read import read_graph_from_json
from src.utils import get_roots
from src.experiment import run_optimization_experiment
from src.draw import visualize_roots

os.makedirs("../../results", exist_ok=True)

In [ ]:
G = read_graph_from_json("../../data/IA_county.json", state="IA")

print(
    f"Iowa county graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges"
)

In [ ]:
K = 4
roots = get_roots(G, k=K)
print(f"Roots: {roots}")

fig = visualize_roots(G, roots, title="Iowa Counties — Euclidean M Optimization Roots (k=4)")
plt.show()

### Experiment parameters

We sweep over a range of population deviations, all contiguity formulations, and a representative set of objectives. `distance_metric="euclidean"` sets each edge weight to the Euclidean distance between county centroids (in projected km). These weighted distances feed into the tree, dist, and dag contiguity constraints via Dijkstra shortest paths, giving a geographically-aware Big-M.

We also include `weighted_moi` as an objective, which directly minimises moment-of-inertia using the same Euclidean edge weights.

In [ ]:
DEVIATIONS   = [50, 100, 200, 500, 1000]
CONTIGUITY   = ["tree", "dist", "dag", "cut"]
OBJECTIVES   = ["hop_moi", "euclidean_moi", "weighted_moi", "cut_edges"]
TIME_LIMIT   = 3600

total = len(DEVIATIONS) * len(CONTIGUITY) * len(OBJECTIVES)
print(f"Deviations : {DEVIATIONS}")
print(f"Contiguity : {CONTIGUITY}")
print(f"Objectives : {OBJECTIVES}")
print(f"Total experiments to run: {total}")

In [ ]:
df = run_optimization_experiment(
    G_base=G,
    deviations=DEVIATIONS,
    contiguity_models=CONTIGUITY,
    objectives=OBJECTIVES,
    k=K,
    distance_metric="euclidean",  # Euclidean M: edge weight = Euclidean distance between centroids
    time_limit=TIME_LIMIT,
    results_file="../results/IA_county_optimization_euclidean_M.csv",
)

In [ ]:
df